# 04 — Label Construction

**Goals**
- Find each cycle’s end time (first timestamp where moisture ≤ threshold)
- Compute time-remaining for every row
- Map time-remaining → urgency bucket (`<24h` / `24-48h` / `>48h`)
- Produce `labeled_dataset.csv`


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
MOISTURE_THRESHOLD = 25.0   # cycle ends when moisture first drops to or below this

features = pd.read_csv(PROCESSED_DIR / 'features.csv', parse_dates=['timestamp'])
manifest = pd.read_csv(PROCESSED_DIR / 'cycle_manifest.csv', parse_dates=['start_time', 'end_time'])
valid_ids = manifest[manifest['valid']]['cycle_id'].tolist()


In [ ]:
# Determine end time per cycle
cycle_ends = {}
for f in sorted(RAW_DIR.glob('*.csv')):
    df = pd.read_csv(f, parse_dates=['timestamp']).sort_values('timestamp')
    cid = df['cycle_id'].iloc[0]
    if cid not in valid_ids:
        continue
    crossed = df[df['soil_moisture_pct'] <= MOISTURE_THRESHOLD]
    if len(crossed) > 0:
        end_ts = crossed['timestamp'].iloc[0]
    else:
        end_ts = df['timestamp'].iloc[-1]
    cycle_ends[cid] = end_ts
    print(f'{cid}: end={end_ts}   final_moisture={df.soil_moisture_pct.iloc[-1]:.2f}')


In [ ]:
def urgency_bucket(hours):
    if hours < 24:
        return '<24h'
    elif hours <= 48:
        return '24-48h'
    else:
        return '>48h'

features = features.copy()
features['cycle_end'] = features['cycle_id'].map(cycle_ends)
features['time_remaining_h'] = (
    (features['cycle_end'] - features['timestamp']).dt.total_seconds() / 3600.0
).clip(lower=0)
features['urgency'] = features['time_remaining_h'].apply(urgency_bucket)

features[['timestamp', 'cycle_id', 'soil_moisture_pct', 'time_remaining_h', 'urgency']].head(10)


In [ ]:
print('Urgency value counts:')
print(features['urgency'].value_counts())
print()
print(features.groupby(['condition', 'urgency']).size().unstack(fill_value=0))


In [ ]:
# Final labeled dataset
label_cols = [
    'timestamp', 'cycle_id', 'condition',
    'temperature_C', 'humidity_pct', 'soil_moisture_pct',
    'moisture_trend', 'light_lux', 'hour_of_day',
    'time_remaining_h', 'urgency'
]
labeled = features[label_cols].copy()
labeled.to_csv(PROCESSED_DIR / 'labeled_dataset.csv', index=False)
print(f'Saved labeled_dataset.csv with {len(labeled)} rows')
